In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
DATA_DIR = "data"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
 
 
def savefig(name):
    path = os.path.join(OUTPUT_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=120)
    plt.close()
    print(f"  -> grafico salvato in: {path}")


In [ ]:
print("=" * 70)
print("1. CARICAMENTO DATI")
print("=" * 70)
 
train_path = os.path.join(DATA_DIR, "train.csv")
store_path = os.path.join(DATA_DIR, "store.csv")
 
train = pd.read_csv(train_path, parse_dates=["Date"], low_memory=False)
store = pd.read_csv(store_path)
 
print(f"train.csv:  {train.shape[0]:,} righe, {train.shape[1]} colonne")
print(f"store.csv:  {store.shape[0]:,} righe, {store.shape[1]} colonne")
 



In [ ]:
print("\n" + "=" * 70)
print("2. MERGE train + store (su 'Store')")
print("=" * 70)
 
df = train.merge(store, how="left", on="Store")
print(f"Dataset unito: {df.shape[0]:,} righe, {df.shape[1]} colonne")
print("\nColonne disponibili:")
print(list(df.columns))


In [ ]:
print("\n" + "=" * 70)
print("3. STRUTTURA E TIPI DI DATO")
print("=" * 70)
print(df.dtypes)
 
print("\nAnteprima (prime 5 righe):")
print(df.head())
 
print("\nStatistiche descrittive (variabili numeriche):")
print(df.describe())
 
print(f"\nNumero di store unici: {df['Store'].nunique()}")
print(f"Intervallo temporale: {df['Date'].min().date()} -> {df['Date'].max().date()}")


In [ ]:
print("\n" + "=" * 70)
print("4. VALORI MANCANTI")
print("=" * 70)
 
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"n_missing": missing, "pct_missing": missing_pct})
missing_report = missing_report[missing_report["n_missing"] > 0].sort_values(
    "n_missing", ascending=False
)
print(missing_report if not missing_report.empty else "Nessun valore mancante.")
 


In [ ]:
print("\n" + "=" * 70)
print("5. ANALISI DEL TARGET (Sales)")
print("=" * 70)
 
n_closed = (df["Open"] == 0).sum()
n_zero_sales = (df["Sales"] == 0).sum()
print(f"Righe con negozio chiuso (Open == 0): {n_closed:,} ({n_closed/len(df)*100:.1f}%)")
print(f"Righe con Sales == 0: {n_zero_sales:,} ({n_zero_sales/len(df)*100:.1f}%)")
 
open_df = df[df["Open"] == 1]
print("\nStatistiche di Sales SOLO sui giorni di apertura:")
print(open_df["Sales"].describe())
 
plt.figure(figsize=(8, 5))
plt.hist(open_df["Sales"], bins=100)
plt.title("Distribuzione di Sales (negozi aperti)")
plt.xlabel("Sales")
plt.ylabel("Frequenza")
savefig("01_hist_sales.png")
 
plt.figure(figsize=(8, 5))
plt.hist(np.log1p(open_df["Sales"]), bins=100)
plt.title("Distribuzione di log(1 + Sales) (negozi aperti)")
plt.xlabel("log(1 + Sales)")
plt.ylabel("Frequenza")
savefig("02_hist_log_sales.png")


In [ ]:
print("\n" + "=" * 70)
print("6. TREND TEMPORALI")
print("=" * 70)
 
# Vendite medie per giorno della settimana
sales_by_dow = open_df.groupby("DayOfWeek")["Sales"].mean()
plt.figure(figsize=(8, 5))
sales_by_dow.plot(kind="bar")
plt.title("Vendite medie per giorno della settimana")
plt.xlabel("Giorno della settimana (1=Lun ... 7=Dom)")
plt.ylabel("Sales medie")
savefig("03_sales_by_dayofweek.png")
 
# Vendite medie mensili nel tempo
df_time = open_df.copy()
df_time["YearMonth"] = df_time["Date"].dt.to_period("M")
sales_by_month = df_time.groupby("YearMonth")["Sales"].mean()
plt.figure(figsize=(12, 5))
sales_by_month.plot()
plt.title("Vendite medie mensili nel tempo")
plt.xlabel("Anno-Mese")
plt.ylabel("Sales medie")
savefig("04_sales_by_month.png")
 
# Effetto promo
sales_by_promo = open_df.groupby("Promo")["Sales"].mean()
print("\nVendite medie in base a Promo (0=no, 1=si):")
print(sales_by_promo)
 
plt.figure(figsize=(6, 5))
sales_by_promo.plot(kind="bar")
plt.title("Vendite medie: giorni con/senza promozione")
plt.xlabel("Promo")
plt.ylabel("Sales medie")
savefig("05_sales_by_promo.png")
 
# Effetto StateHoliday
sales_by_holiday = open_df.groupby("StateHoliday")["Sales"].mean()
print("\nVendite medie in base a StateHoliday:")
print(sales_by_holiday)


In [ ]:
print("\n" + "=" * 70)
print("7. ANALISI PER TIPO DI NEGOZIO (StoreType / Assortment)")
print("=" * 70)
 
print("\nDistribuzione StoreType:")
print(df["StoreType"].value_counts())
 
print("\nDistribuzione Assortment:")
print(df["Assortment"].value_counts())
 
sales_by_storetype = open_df.groupby("StoreType")["Sales"].mean().sort_values(ascending=False)
print("\nVendite medie per StoreType:")
print(sales_by_storetype)
 
plt.figure(figsize=(6, 5))
sales_by_storetype.plot(kind="bar")
plt.title("Vendite medie per StoreType")
plt.xlabel("StoreType")
plt.ylabel("Sales medie")
savefig("06_sales_by_storetype.png")


In [ ]:
print("\n" + "=" * 70)
print("8. CORRELAZIONI (feature numeriche)")
print("=" * 70)
 
numeric_cols = open_df.select_dtypes(include=[np.number]).columns
corr = open_df[numeric_cols].corr()
print(corr["Sales"].sort_values(ascending=False))
 
plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlazione")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Matrice di correlazione")
savefig("07_correlation_matrix.png")
 
print("\n" + "=" * 70)
print("DATA UNDERSTANDING COMPLETATO")
print("=" * 70)
print(f"Tutti i grafici sono stati salvati nella cartella '{OUTPUT_DIR}/'.")
 
